In [ ]:
import cv2
import pandas as pd
import supervision as sv
import os
from datetime import datetime
from ultralytics import YOLO
import pymysql
import mysql_account_info as sql_info_JCON
import PySimpleGUI as sg
import requests

In [ ]:
#資料庫確認是否有已存在的資料表
def db_check(table_name):
    event, values = window1.read(timeout=30)
    #資料庫連線設定 預設是使用本機端 遠端連線需再調整
    db = pymysql.connect(host='localhost', port=3306, user='root', passwd=sql_info.password, db='test', charset='utf8')
    cursor=db.cursor()
    sql1="USE test;"
    cursor.execute(sql1)
    cursor.execute('show tables;')
    table_list=[a[0] for a in cursor.fetchall()]

    if table_name not in table_list:
        sql2="CREATE TABLE "+table_name+"(event_class CHAR(10),event_time DATETIME,Webcam CHAR(10));"
        cursor.execute(sql2)
        cmd=values['-MULTILINE KEY-']+'\n'+"["+str(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))+"]"+"data base could't find named:"+table_name+" table and system has created a new table"
        window1.Element('-MULTILINE KEY-').Update(cmd)
    else:
        cmd=values['-MULTILINE KEY-']+'\n'+"["+str(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))+"]"+"named "+table_name+" table already exists and connection status is normal."
        window1.Element('-MULTILINE KEY-').Update(cmd)

    db.commit()

In [ ]:
#opencv圖像設置以及處理
def cap_setting(path,width,height):
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)

    return cap

#圖像資料型態轉換 yt爬的不用管這裡在幹嘛
def frame_convertion(cap,conf_lower_limit,model,box_annotator):
    ret,frame = cap.read()

    result = model(frame,agnostic_nms=True,conf=conf_lower_limit)[0] #conf=機率多少以上才標框
    detections = sv.Detections.from_ultralytics(result)
    labels = [
        f"{model.model.names[class_id]} {confidence:0.2f}"
        for _, _,confidence, class_id,_
        in detections
    ]

    img = box_annotator.annotate(
        scene=frame,
        detections=detections,
        labels=labels
    )

    return result,img

#將yolo標註的信息轉換成int組成的list
def detect_info(result):
    detect_list=[]
    detect_list=list(map(int,result.boxes.cls.tolist()))
    return detect_list

In [ ]:
#資料庫寫入動作
def db_insert(table_name,Now,cap_count):
    #計時30毫秒刷新UI
    event, values = window1.read(timeout=30)

    try:
        db = pymysql.connect(host='localhost', port=3306, user='root', passwd=sql_info.password, db='test', charset='utf8')
        cursor=db.cursor()
        sql1="USE test;"
        cursor.execute(sql1)
        sql_event="INSERT INTO "+table_name+"(event_class, event_time, Webcam) VALUES ('abnormal','"+str(Now) +"','Webcam"+str(cap_count)+"');"
        cursor.execute(sql_event)
        db.commit()
        cmd=values['-MULTILINE KEY-']+'\n'+"["+str(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))+"]"+'Insert operation success'
        window1.Element('-MULTILINE KEY-').Update(cmd)
    except:
        cmd=values['-MULTILINE KEY-']+'\n'+"["+str(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))+"]"+"Error: Could not make connection to the MySQL database"
        window1.Element('-MULTILINE KEY-').Update(cmd)

In [ ]:
def imgresize(a):
    imgbytes = cv2.resize(a, (480,360))
    imgbytes = cv2.imencode('.png', imgbytes)[1].tobytes()
    return imgbytes

In [ ]:
def model_operate(table_name="",cap_count=0,cap_path=0):

    #讀取model 路徑前面的r只是翻轉'/'而已
    model = YOLO(r'D:\jcon_n_net_an\weights\best.pt')
    #####################################################
    #讀取影片並設定Size以及影片數量
    cap=cap_setting(cap_path,width=608,height=352)

    #事件觸發計數器
    cap_detect_count=0
    #事件結算暫存器
    global cap_detect_check
    cap_detect_check=0
    #觸發時間記錄器
    time_detect=60

    #Line notify setting
    #參考資料:https://officeguide.cc/python-line-notify-send-messages-images-tutorial-examples/

    token = 'MewVp9rdklBcOTcxkw0iodMYcQyp8WYd73skkIf7RRQ'
    message = 'event: abnormal'
    headers = { "Authorization": "Bearer " + token }
    data = { 'message': message }

    box_annotator = sv.BoxAnnotator(thickness=2, text_thickness=2, text_scale=1)
    ####################################################
    while True:
        event, values = window1.read(timeout=7)
        if event == sg.WIN_CLOSED:break

        Now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        ####################################################
        #將yolo predict的結果轉換成sv形式
        result,img=frame_convertion(cap,conf_lower_limit=0.1,model=model,box_annotator=box_annotator)
        ####################################################
        #使用按鈕處理不同場域
        window1["-V-IMG-"].update(data=imgresize(img))

        ##################################################
        #opencv必須設定延遲，沒設定預覽會變成黑畫面
        ##################################################
        detect_list=detect_info(result)
        ##################################################
        #第一次偵測到abnormal將偵測到的時間紀錄，並且abnormal計數器+1
        if (1 in detect_list)&(cap_detect_count==0):
            cap_detect_count+=1
            time_detect=int(datetime.now().strftime("%S"))
            cmd=values['-MULTILINE KEY-']+'\n'+"["+str(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))+"]"+"time_counter_start"
            window1.Element('-MULTILINE KEY-').Update(cmd)
        ##################################################
        #非第一次偵測到只需將abnormal計數器+1
        elif (1 in detect_list):
            cap_detect_count+=1
        ##################################################

        #cv2.waitKey(10)

        #當第一次偵測到的時間+10秒=現在時間，把abnormal計數器歸零
        if  (time_detect+10)==(int(datetime.now().strftime("%S"))):
            cap_detect_check=cap_detect_count
            cap_detect_count=0
            time_detect=60
            cmd=values['-MULTILINE KEY-']+'\n'+"["+str(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))+"]"+"time_counter_stop"
            window1.Element('-MULTILINE KEY-').Update(cmd)
        ##################################################
            #進行結算，若結算後abnormal次數>=2，則寫進資料庫並啟動LINE通知
            if cap_detect_check>=2:
                requests.post("https://notify-api.line.me/api/notify",headers = headers, data = data)
                db_insert(table_name,Now,cap_count)
                cv2.waitKey(10)
                cmd=values['-MULTILINE KEY-']+'\n'+"["+str(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))+"]"+"data insert successfully and post the msg with Line notify"
                window1.Element('-MULTILINE KEY-').Update(cmd)

            ##################################################
            #break;



        cv2.waitKey(1)

    cap.release()
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    window1.close()

In [ ]:
def UI_layout():
    #####################################################
    #UI版型
    sg.theme('LightGreen7')
    #Light Color #LightGreen7

    #初始UI預覽畫面窗格的圖片設定
    imgbytes = cv2.imread(r'D:\UI_test\Image1.png')
    imgbytes = cv2.resize(imgbytes, (480,360))
    imgbytes = cv2.imencode('.png', imgbytes)[1].tobytes()

    m1 = sg.Button('monitor1', size = 8, key = "-IMAGE-1-")
    m2 = sg.Button('monitor2', size = 8, key = "-IMAGE-2-")
    m3 = sg.Button('monitor3', size = 8, key = "-IMAGE-3-")
    m4 = sg.Button('monitor4', size = 8, key = "-IMAGE-4-")
    #list_view = sg.Listbox([1,2,3,4], size = (20,16), enable_events=True, key = "-V-CHOOSE-")
    #video_image = sg.Image('Image1.png',size = (400,400), key = "-V-IMG-")
    video_image = sg.Image(data=imgbytes,size=(480,360),key = "-V-IMG-")
    multi = sg.Multiline("["+str(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))+"]"'System operation', size = (50,20),font = ('宋体',12),key = '-MULTILINE KEY-',disabled=True)


    layout = [
            [sg.Text('監視器畫面',font = ('宋体',12)),sg.Text('',size=67)],
            [video_image, multi],
            [m1,m2,m3,m4]
              ]

    global window1
    window1 = sg.Window("工廠智慧巡檢系統", layout)
    #####################################################

In [ ]:
if __name__ == "__main__":
    table_name="check_test"
    #初始UI輸出
    UI_layout()
    #檢查資料庫連線並且查看是否有叫做check_test的資料表存在
    db_check(table_name)
    #model以及主程式運作
    model_operate(table_name,1,r'D:\data_feet_jcon\test_video\jcon_test1.mp4')


0: 352x608 1 normal, 105.5ms
Speed: 7.3ms preprocess, 105.5ms inference, 3.9ms postprocess per image at shape (1, 3, 352, 608)

0: 352x608 1 normal, 47.6ms
Speed: 4.5ms preprocess, 47.6ms inference, 3.1ms postprocess per image at shape (1, 3, 352, 608)

0: 352x608 1 normal, 47.7ms
Speed: 3.7ms preprocess, 47.7ms inference, 4.5ms postprocess per image at shape (1, 3, 352, 608)

0: 352x608 1 normal, 47.8ms
Speed: 2.8ms preprocess, 47.8ms inference, 3.4ms postprocess per image at shape (1, 3, 352, 608)

0: 352x608 1 normal, 47.9ms
Speed: 4.0ms preprocess, 47.9ms inference, 4.0ms postprocess per image at shape (1, 3, 352, 608)

0: 352x608 1 normal, 1 abnormal, 47.6ms
Speed: 4.0ms preprocess, 47.6ms inference, 6.1ms postprocess per image at shape (1, 3, 352, 608)

0: 352x608 1 normal, 48.2ms
Speed: 3.4ms preprocess, 48.2ms inference, 3.9ms postprocess per image at shape (1, 3, 352, 608)

0: 352x608 1 normal, 47.8ms
Speed: 3.0ms preprocess, 47.8ms inference, 4.0ms postprocess per image at s